# M3 · Load dimensions

**Goal:** run `sql/01_dim_tenant.sql` through `sql/05_dim_date_hour.sql` in order. These are full-refresh idempotent upserts — re-running does not duplicate rows.

This is the data-loading half of what the old `bootstrap` mode did.

**Exit criterion:** row counts match (5 tenants, 638 vehicles, 633 devices, 294 drivers, ~3287 dates, 24 hour-bands).


In [2]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — dimension SQL files

In [3]:
from accent_fleet.db.sql_loader import load_sql
DIM_FILES = [
    '01_dim_tenant.sql',
    '02_dim_vehicle.sql',
    '03_dim_device.sql',
    '04_dim_driver.sql',
    '05_dim_date_hour.sql',
]
for f in DIM_FILES:
    head = load_sql(f).splitlines()[:5]
    print(f"\n-- {f}")
    for l in head:
        print(l)



-- 01_dim_tenant.sql
-- =============================================================================
-- 01_dim_tenant.sql
-- =============================================================================
-- Tenant dimension. Idempotent upsert from distinct tenant_ids in staging.
-- Full-scan is fine — there are 5 tenants.

-- 02_dim_vehicle.sql
-- =============================================================================
-- 02_dim_vehicle.sql
-- =============================================================================
-- Vehicle dimension. Normalises make (C7), derives vehicle class, handles
-- max_speed = 0 -> NULL. Idempotent via upsert on natural key.

-- 03_dim_device.sql
-- =============================================================================
-- 03_dim_device.sql
-- =============================================================================
-- Device dimension. Enforces C7 via INNER JOIN to dim_vehicle — any device
-- without a valid vehicle linkage is excluded (

## 3. Execute — via the shared helper (same code path as production)

In [4]:
import time
from accent_fleet.transforms import refresh_all_dimensions
t0 = time.time()
refresh_all_dimensions()
print(f"Dimensions refreshed in {time.time()-t0:.2f}s")


2026-04-29 17:10:43 [info     ] dim_refresh.start              file=01_dim_tenant.sql
2026-04-29 17:10:45 [info     ] dim_refresh.done               file=01_dim_tenant.sql
2026-04-29 17:10:45 [info     ] dim_refresh.start              file=02_dim_vehicle.sql
2026-04-29 17:10:45 [info     ] dim_refresh.done               file=02_dim_vehicle.sql
2026-04-29 17:10:45 [info     ] dim_refresh.start              file=03_dim_device.sql
2026-04-29 17:10:46 [info     ] dim_refresh.done               file=03_dim_device.sql
2026-04-29 17:10:46 [info     ] dim_refresh.start              file=04_dim_driver.sql
2026-04-29 17:10:46 [info     ] dim_refresh.done               file=04_dim_driver.sql
2026-04-29 17:10:46 [info     ] dim_refresh.start              file=05_dim_date_hour.sql
2026-04-29 17:10:47 [info     ] dim_refresh.done               file=05_dim_date_hour.sql
2026-04-29 17:10:47 [info     ] dim_refresh.start              file=07_bridge_device_driver_load.sql
2026-04-29 17:10:47 [info     ]

## 4. Inspect

In [5]:
import pandas as pd
EXPECTED = {'dim_tenant': 5, 'dim_vehicle': 638, 'dim_device': 633, 'dim_driver': 294, 'dim_hour_band': 24}
rows = []
with get_engine().connect() as conn:
    for t in ['dim_tenant','dim_vehicle','dim_device','dim_driver','dim_date','dim_hour_band']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM warehouse.{t}')).scalar_one()
        rows.append({'table': t, 'rows': n})
df = pd.DataFrame(rows); display(df)
for t, expected in EXPECTED.items():
    got = int(df.loc[df['table']==t, 'rows'].iloc[0])
    assert got == expected or abs(got-expected) <= 5, f"{t}: expected ~{expected}, got {got}"
print("OK — dimensions populated.")


,table,rows
0,dim_tenant,5
1,dim_vehicle,638
2,dim_device,633
3,dim_driver,294
4,dim_date,3287
5,dim_hour_band,24


OK — dimensions populated.


In [6]:
# Sample rows for visual check
import pandas as pd
with get_engine().connect() as conn:
    for t in ['dim_tenant','dim_vehicle','dim_device','dim_driver']:
        print(f"\n=== warehouse.{t} (sample) ===")
        display(pd.read_sql(text(f'SELECT * FROM warehouse.{t} LIMIT 5'), conn))



=== warehouse.dim_tenant (sample) ===


,tenant_id,tenant_label,first_seen_at,last_seen_at
0,1787,tenant_1787,2026-04-21 14:32:45.002498+00:00,2026-04-29 16:10:44.160398+00:00
1,238,tenant_238,2026-04-21 14:32:45.002498+00:00,2026-04-29 16:10:44.160398+00:00
2,235,tenant_235,2026-04-21 14:32:45.002498+00:00,2026-04-29 16:10:44.160398+00:00
3,7486,tenant_7486,2026-04-21 14:32:45.002498+00:00,2026-04-29 16:10:44.160398+00:00
4,264,tenant_264,2026-04-21 14:32:45.002498+00:00,2026-04-29 16:10:44.160398+00:00



=== warehouse.dim_vehicle (sample) ===


,vehicle_sk,tenant_id,vehicule_id,mark_raw,mark_clean,vehicle_class,matricule,max_speed_setting,category,model,status,_loaded_at
0,46,7486,78,Ford (6 ch),Ford (6 Ch),unknown,3954 TU 195,120,None,None,0,2026-04-29 16:10:44.160398+00:00
1,47,7486,79,Isuzu ( 8 ch ),Isuzu ( 8 Ch ),unknown,6116 TU 195,120,None,None,0,2026-04-29 16:10:44.160398+00:00
2,48,7486,80,IVECO,Iveco,heavy,1789 TU 242,110,None,None,0,2026-04-29 16:10:44.160398+00:00
3,49,7486,82,SPORTAGE ( Kia),Sportage ( Kia),unknown,6831 TU 196,130,None,None,0,2026-04-29 16:10:44.160398+00:00
4,50,7486,83,FIAT DOBLO,Fiat Doblo,unknown,7867 TU 255,110,None,None,0,2026-04-29 16:10:44.160398+00:00



=== warehouse.dim_device (sample) ===


,device_sk,device_id,tenant_id,vehicle_sk,imei,serial,_loaded_at
0,118,425218,235,435,None,None,2026-04-29 16:10:44.160398+00:00
1,119,425219,235,436,None,None,2026-04-29 16:10:44.160398+00:00
2,120,425220,235,437,None,None,2026-04-29 16:10:44.160398+00:00
3,121,425221,235,438,None,None,2026-04-29 16:10:44.160398+00:00
4,122,425222,235,317,None,None,2026-04-29 16:10:44.160398+00:00



=== warehouse.dim_driver (sample) ===


,driver_sk,driver_id,tenant_id,first_name,last_name,is_medically_fit,has_training,is_safe_driver,is_authorized,_loaded_at
0,31,17,1787,JELLITI,NOUREDDINE,False,False,False,False,2026-04-29 16:10:44.160398+00:00
1,32,18,1787,RIDHA,AZZOUZI,False,False,False,False,2026-04-29 16:10:44.160398+00:00
2,33,19,1787,MED,ALI HADJRI,False,False,False,False,2026-04-29 16:10:44.160398+00:00
3,34,20,1787,TIJANI,BOUKIL,False,False,False,False,2026-04-29 16:10:44.160398+00:00
4,35,21,1787,MEJED TRIMECH,NaN,False,False,False,False,2026-04-29 16:10:44.160398+00:00
